In [1]:
import boto3
import pandas as pd
import sagemaker

In [3]:
from sagemaker.core.helper.session_helper import Session, get_execution_role

session = Session()
role = get_execution_role()
# bucket = session.default_bucket()
bucket = "bkt-deloitte-aug26-hyd"
print("Bucket:", bucket)
print("Role:", role)

Bucket: bkt-deloitte-aug26-hyd
Role: arn:aws:iam::781843541123:role/service-role/AmazonSageMaker-ExecutionRole-20260826T191955


In [9]:
session.upload_data(
    path="FreshMart_Customer_Membership.csv",
    bucket=bucket,
    key_prefix="raw-data"
)

's3://bkt-deloitte-aug26-hyd/raw-data/FreshMart_Customer_Membership.csv'

## Data Preparation:

In [10]:
df = pd.read_csv(f"s3://{bucket}/raw-data/FreshMart_Customer_Membership.csv")
df.head()

,Customer_ID,Customer_Name,Age,Gender,Income,Orders_Last_Year,Average_Order_Value,City,App_Usage_Hours,Membership_Years,Premium_Member
0,C000001,Aanya Verma,43,Male,97822.0,13.0,3086.0,Surat,5.3,3.0,0.0
1,C000002,Neha Kumar,22,Male,92126.0,13.0,3929.0,Hyderabad,3.9,10.0,0.0
2,C000003,Vivaan Das,55,Female,86821.0,11.0,2598.0,Kolkata,4.4,2.0,0.0
3,C000004,Meera Kulkarni,53,Male,95948.0,14.0,3810.0,Indore,5.6,1.0,1.0
4,C000005,Nisha Nair,49,Female,74764.0,3.0,5011.0,Hyderabad,3.9,12.0,0.0


In [11]:
df.shape

(33792, 11)

In [12]:
df.isnull().sum()

Customer_ID              0
Customer_Name            0
Age                      0
Gender                   0
Income                 677
Orders_Last_Year         1
Average_Order_Value    641
City                   662
App_Usage_Hours        673
Membership_Years         1
Premium_Member           1
dtype: int64

In [13]:
df.duplicated().sum()

0

In [14]:
df = df.drop_duplicates()

In [15]:
df.duplicated().sum()

0

In [30]:
df = df.dropna(subset=["Premium_Member"])
X = df.drop(
    columns=[
        "Premium_Member",
        "Customer_ID",
        "Customer_Name"
    ]
)
y = df["Premium_Member"]

In [31]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing y:", y.isna().sum())
print(y.value_counts())

X shape: (33791, 8)
y shape: (33791,)
Missing y: 0
Premium_Member
0.0    25348
1.0     8443
Name: count, dtype: int64


## Train/Test Split

In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

## Handle Missing Values + Encoding

In [33]:
cat_cols = X.select_dtypes("object").columns
num_cols = X.select_dtypes(exclude="object").columns

In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        num_cols
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        cat_cols
    )
])

In [35]:
print("y_train nulls:", y_train.isna().sum())
print("y_test nulls:", y_test.isna().sum())

print(y_train.value_counts(dropna=False))

y_train nulls: 0
y_test nulls: 0
Premium_Member
0.0    20201
1.0     6831
Name: count, dtype: int64


## Model Experiments

1) Model V1 → Logistic Regression

In [36]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

lr = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

2) Random Forest

In [37]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

# Compare Models

In [38]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def metrics(y, pred):
    return [
        accuracy_score(y, pred),
        precision_score(y, pred, zero_division=0),
        recall_score(y, pred, zero_division=0),
        f1_score(y, pred, zero_division=0)
    ]

results = pd.DataFrame([
    ["Logistic Regression", "V1", *metrics(y_test, lr_pred)],
    ["Random Forest", "V2", *metrics(y_test, rf_pred)]
], columns=[
    "Model",
    "Version",
    "Accuracy",
    "Precision",
    "Recall",
    "F1"
])
results

,Model,Version,Accuracy,Precision,Recall,F1
0,Logistic Regression,V1,0.759284,0.317073,0.008065,0.015729
1,Random Forest,V2,0.755733,0.454545,0.120968,0.191083


# Select Best Model

In [39]:
best_model_name = results.loc[
    results["F1"].idxmax(),
    "Model"
]
print("Candidate Model:", best_model_name)

Candidate Model: Random Forest


In [40]:
candidate = rf if best_model_name == "Random Forest" else lr
candidate

,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Save Experiment Results

In [43]:
results.to_csv(
    "experiment_results.csv",
    index=False
)

In [44]:
session.upload_data(
    path="experiment_results.csv",
    bucket=bucket,
    key_prefix="experiments"
)

's3://bkt-deloitte-aug26-hyd/experiments/experiment_results.csv'

# Save Models

In [45]:
import joblib

joblib.dump(lr, "model_v1.joblib")
joblib.dump(rf, "model_v2.joblib")

['model_v2.joblib']

In [46]:
session.upload_data(
    path="model_v1.joblib",
    bucket=bucket,
    key_prefix="models"
)

session.upload_data(
    path="model_v2.joblib",
    bucket=bucket,
    key_prefix="models"
)

's3://bkt-deloitte-aug26-hyd/models/model_v2.joblib'